# NWP to Zarr: Comparison & Validation

This notebook validates the generated **Zarr** stores against the original **FA** files.

### Objectives:
1. **Temperature (2t)**: Verify K to °C conversion.
2. **Precipitation (tp)**: Verify sliding decumulation (e.g. RR3h(T) = Acc(T) - Acc(T-3)).
3. **Time Slicing**: Verify Zarr groups start at the first valid leadtime (H03 for 3h, H06 for 6h).

In [ ]:
import os
import epygram
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

epygram.init_env()

# CONFIGURATION
RUN = "2026030100"
MODEL = "arome"
FA_DIR = Path(f"/fennecData/data/chprod/{MODEL.upper()}/FULLPOS/2026/03/01/r00")
ZARR_ROOT = Path(f"/fennecData/home/pnt/metview/Prod/scr2/zarr/{MODEL}/{RUN}")

FA_H00 = FA_DIR / "FULLPOS_2026030100_0000"
FA_H03 = FA_DIR / "FULLPOS_2026030100_0003"
FA_H06 = FA_DIR / "FULLPOS_2026030100_0006"

print(f"Validation for {MODEL} run {RUN}")

## 1. Temperature (2t) Validation
Checking if Kelvin to Celsius conversion is correct.

In [ ]:
def compare_2t(fa_path, zarr_path):
    # Load FA
    r = epygram.formats.resource(str(fa_path), 'r')
    fa_data = r.readfield('CLSTEMPERATURE').getdata() - 273.15
    r.close()
    
    # Load Zarr
    ds = xr.open_zarr(zarr_path)
    zarr_data = ds['2t'].isel(time=0).values
    
    diff = fa_data - zarr_data
    max_diff = np.abs(diff).max()
    
    print(f"Temperature 2t Comparison:")
    print(f"  - Max Difference: {max_diff:.6e} °C")
    print(f"  - Mean Error:     {np.mean(diff):.6e} °C")
    
    if max_diff < 1e-4: 
        print("  ✅ SUCCESS: Data matches (within float32 precision)")
    else:
        print("  ❌ FAILURE: Significant difference detected")

compare_2t(FA_H00, ZARR_ROOT / "surface.zarr")

## 2. Sliding Accumulation Validation
Checking if sliding Zarr value matches the difference between FA files.
Example: In `surface_3h.zarr`, index 0 is H03. It must match `FA_H03 - FA_H00`.

In [ ]:
def compare_acc(fa_path_1, fa_path_2, zarr_group, var_name='tp', time_idx=0):
    print(f"Validating {var_name} in {zarr_group} at time index {time_idx}...")
    
    # 1. FA Difference
    fa_id = 'SURFACCPLUIE' if var_name == 'tp' else 'SURFACCNEIGE'
    
    r1 = epygram.formats.resource(str(fa_path_1), 'r')
    val1 = r1.readfield(fa_id).getdata()
    r1.close()
    
    r2 = epygram.formats.resource(str(fa_path_2), 'r')
    val2 = r2.readfield(fa_id).getdata()
    r2.close()
    
    fa_diff = val2 - val1
    
    # 2. Zarr Value
    ds = xr.open_zarr(ZARR_ROOT / f"{zarr_group}.zarr")
    # TIME Sliced: Index 0 is the first valid leadtime (e.g. H03 for 3h)
    zarr_val = ds[var_name].isel(time=time_idx).values
    
    print(f"  - Validating leadtime: {ds.time.isel(time=time_idx).values}")
    
    diff = fa_diff - zarr_val
    max_err = np.nanmax(np.abs(diff))
    
    print(f"  - FA Diff Max:           {fa_diff.max():.4f}")
    print(f"  - Zarr Part Max:         {np.nanmax(zarr_val):.4f}")
    print(f"  - Max Comparison Error:  {max_err:.6e}")
    
    if max_err < 1e-4:
        print(f"  ✅ SUCCESS: {var_name} sliding accumulation is correct")
    else:
        print(f"  ❌ FAILURE: {var_name} mismatch or index error")

print("--- 3H Group ---")
compare_acc(FA_H00, FA_H03, "surface_3h", "tp", time_idx=0) # H03
compare_acc(FA_H03, FA_H06, "surface_3h", "tp", time_idx=3) # H06 in 3h group is index 3 (H03, H04, H05, H06)

print("\n--- 6H Group ---")
compare_acc(FA_H00, FA_H06, "surface_6h", "tp", time_idx=0) # H06


## 3. Visualization: Detailed Field Plot

In [ ]:
def plot_comparison(fa_path_1, fa_path_2, zarr_group, var_name='tp', time_idx=0):
    # 1. FA Difference
    fa_id = 'SURFACCPLUIE' if var_name == 'tp' else 'SURFACCNEIGE'
    val1 = epygram.formats.resource(str(fa_path_1), 'r').readfield(fa_id).getdata()
    val2 = epygram.formats.resource(str(fa_path_2), 'r').readfield(fa_id).getdata()
    fa_diff = val2 - val1
    
    # 2. Zarr Value
    ds = xr.open_zarr(ZARR_ROOT / f"{zarr_group}.zarr")
    zarr_val = ds[var_name].isel(time=time_idx).values
    
    # Plotting
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    
    im1 = axes[0].imshow(fa_diff, cmap="Blues")
    axes[0].set_title(f"FA Diff (H03-H00)")
    plt.colorbar(im1, ax=axes[0])
    
    im2 = axes[1].imshow(zarr_val, cmap="Blues")
    axes[1].set_title(f"Zarr Value (index {time_idx})")
    plt.colorbar(im2, ax=axes[1])
    
    plt.suptitle(f"Comparison: {var_name} in {zarr_group}")
    plt.show()

plot_comparison(FA_H00, FA_H03, "surface_3h", "tp", time_idx=0)

## 4. Multi-step Plot (Starting from H03)

In [1]:
ds_3h = xr.open_zarr(ZARR_ROOT / "surface_3h.zarr")
ds_3h['tp'].isel(time=slice(0, 10)).plot(col="time", col_wrap=4, cmap="Blues")
plt.suptitle("Successive Sliding 3h Preciprations (tp) from Zarr")
plt.show()

NameError: name 'xr' is not defined